# R7: optical-loop and RF-chain phase FRA

Two measurements share the primary PLL analyzer; run one at a time. Both use a swept sine with long coherent averaging, 10 Hz–1 MHz.

* **`closed`**: keep the laser locked normally. Inject a small sine after the FPGA error calculation/IIR, on DAC229/1 shared by the analog controllers. Record received phase and total DAC command. This measures the combined feedback loop, including the discriminator, selected FPGA IIR, analog controllers, laser and receiver.
* **`rf`**: disconnect the laser controllers from the driven test setup. Connect **DAC229/2 → RF amplifier chain under test → ADC224/3**. Apply phase modulation to the 800.01 MHz carrier. Error DAC229/1 is held at code zero. Repeat with the RF chain replaced by a direct cable, with suitable attenuation and comparable ADC input level, to separate converter/receiver response from the RF chain.

Use the **R7 RF-FRA** bit/HWH and all three helpers from the same package. This is A7 PLL firmware with feature ID `0x52463701`; the original R7 image lacks this feature. A8 loopback files remain a separate package. No test here uses the ZAD-1+ IF port or its DC calibration.

Run each cell in order. Reprogramming the FPGA interrupts locks. Python-only refresh below preserves an already running lock.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib
import sys
import json
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pynq import Overlay
import xrfdc
import xrfclk

# False: refresh helpers around an existing `ol`, without changing the lock.
# True: configure clocks and download the new matching bit/HWH; interrupts locks.
initialize_hardware = False
bitfile = Path('/home/xilinx/Bitfiles/FreqLocksBranch/overlays/TransportPhaseLockFF.bit')

importlib.invalidate_caches()
import laser_pll_R7 as laser_pll
laser_pll = importlib.reload(laser_pll)
sys.modules['laser_pll'] = laser_pll
import laser_pll_loop_sweep as fra
import laser_pll_phase_fra as pf
fra = importlib.reload(fra)
pf = importlib.reload(pf)
fra.laser_pll = laser_pll
pf.laser_pll = laser_pll
pf.fra = fra

if initialize_hardware:
    custom_lmk_file = '/home/xilinx/jupyter_notebooks/LMK_245_76_ExtRef10.txt'
    custom_regs = []
    with open(custom_lmk_file) as stream:
        for line in stream:
            match = re.search(r'[\t]*(0x[0-9A-F]*)', line.strip(), re.IGNORECASE)
            if match:
                custom_regs.append(int(match.group(1), 16))
    assert custom_regs, 'No clock register words found.'
    if not xrfclk.xrfclk.lmk_devices or not xrfclk.xrfclk.lmx_devices:
        xrfclk.xrfclk._find_devices()
        xrfclk.xrfclk._read_tics_output()
    lmk_chip = xrfclk.xrfclk.lmk_devices[0]['compatible']
    xrfclk.xrfclk._Config[lmk_chip][245.761] = custom_regs
    xrfclk.set_ref_clks(lmk_freq=245.761, lmx_freq=491.52)
    time.sleep(2)
    ol = Overlay(str(bitfile))
    time.sleep(2)
    rfdc = ol.usp_rf_data_converter_0
    for tile in (rfdc.adc_tiles[0], rfdc.dac_tiles[0], rfdc.dac_tiles[1]):
        tile.StartUp()
    time.sleep(1)
else:
    assert 'ol' in globals(), 'Set initialize_hardware=True once, or provide your existing ol.'
    rfdc = ol.usp_rf_data_converter_0

pf.check_helpers(ol)
print('Helpers:', laser_pll.__file__, fra.__file__, pf.__file__, sep='\n')
print('FPGA:', hex(ol.pll_acquisition_status.read(8) >> 24),
      'RF-FRA feature:', hex(ol.pll_rf_source.read(8)))
print('PLL lock: ADC224', rfdc.adc_tiles[0].PLLLockStatus,
      'DAC228', rfdc.dac_tiles[0].PLLLockStatus,
      'DAC229', rfdc.dac_tiles[1].PLLLockStatus)
print('Primary PLL:', laser_pll.acquisition_status(ol))

## Choose the physical measurement

For `closed`, first acquire the normal stage-3/4 optical lock using your existing R7 controls (or the bundled `laser_pll_bringup.ipynb`). Retain the desired tracking gain, FALC/piezo settings and IIR selection. This notebook deliberately does not reacquire or reset a working lock. The measured loop includes both analog branches. Keep all settings fixed during a sweep.

For `rf`, verify the amplifier chain and ADC input power at the selected carrier amplitude before measurement. `setup_rf()` changes the primary PLL to a running receiver with DC-zero output and sets **DAC tile 1/block 2** to the same 800.01 MHz frequency as the receiver's fabric NCO. It leaves DAC229/0, the secondary PLL and the ADC RFDC settings alone. The RF carrier stays on after the probe ends. Reacquire the optical lock after finishing RF tests.

The existing RF tone was I=32767. The default test amplitude below is 8192 (about −12 dB in voltage relative to that); adjust to the RF chain and keep amplifiers/ADC linear.

In [ ]:
measurement_mode = 'closed'  # 'closed' or 'rf'
carrier_hz = 800.01e6
carrier_amplitude_codes = 8192
setup_info = None
if measurement_mode == 'rf':
    setup_info = pf.setup_rf(ol, carrier_hz=carrier_hz,
                            carrier_amplitude_codes=carrier_amplitude_codes)
    print(setup_info)
else:
    pf._ready(ol, 'closed')
    print('Using the current optical lock and controller settings.')

# Record the bench conditions; these are saved with the sweep.
experiment_notes = 'Describe RF wiring, carrier power, FALC switches/gain and piezo gain here.'
amplitude_codes = 32        # closed: peak DAC-code injection (~0.977 mV for nominal 2 Vpp)
phase_amplitude_rad = 0.05  # RF: peak phase modulation; ±0.05 rad, not a DAC voltage
allow_clipping = False     # True retains clipped blocks, explicitly marked unreliable
minimum_measure_s = 1.0    # each coherent block; increase for low SNR
minimum_settle_s = 0.05

## One point before a sweep

Confirm the scope/beat remains well behaved at this amplitude. A probe that breaks lock is too large. RF mode should give a received PM amplitude comparable to the applied 0.05 rad, even if absolute RF gain changes. A half-amplitude repeat helps check linearity.

The diagnostic phase observer unwraps around its value at each arm, with a ±2-turn range. Its overflow or a raw ADC rail flags a block; this is separate from the PLL's much wider feedback winding memory. Large upstream phase slips can invalidate blocks even when the error DAC looks bounded. `allow_clipping=True` keeps such blocks for inspection, but averaging cannot remove nonlinear clipping bias.

In [ ]:
point = pf.measure_point(ol, 1e3, mode=measurement_mode,
                         amplitude_codes=amplitude_codes,
                         phase_amplitude_rad=phase_amplitude_rad,
                         allow_clipping=allow_clipping,
                         minimum_measure_s=minimum_measure_s,
                         minimum_settle_s=minimum_settle_s)
print('Actual frequency:', point['frequency_hz'], 'Hz')
print('Received phase:', point['phase_rad'], 'rad peak')
print('Response:', point['response'],
      'rad/rad' if measurement_mode == 'rf' else 'rad/DAC-code')
print('Diagnostics:', point['diagnostics'])
if measurement_mode == 'closed':
    print('Loop gain L:', (point['stimulus'] - point['drive']) / point['drive'])
    print('Digital error / received phase:', (point['drive'] - point['stimulus']) / point['phase_rad'], 'DAC codes/rad')


## Averaged sweep and saved files

Start with eight independent coherent blocks per frequency. Extend to 24 when the complex response SNR is below 30. This is typically 10–30 minutes for 61 points; low-frequency cycle counts and settling add time. Alternate sine polarity and average complex phasors before taking their ratio. There is no averaging in the feedback path.

Results are saved under `phase_fra_data/` relative to the notebook working directory: one summary CSV, a `_blocks.csv` with every accepted/rejected block and counters, JSON settings/notes, and PNG plots. CSVs are updated after each block/point, so completed data survive an interrupted sweep. A point that cannot meet the target is retained with `reliable=False`; changing amplitudes or clipping overrides never happens automatically.

In [ ]:
frequencies_hz = np.geomspace(10, 1e6, 61)
output_dir = Path('phase_fra_data')
output_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
csv_path = output_dir / f'phase_fra_{measurement_mode}_{stamp}.csv'
metadata = dict(mode=measurement_mode, notes=experiment_notes,
                date=datetime.now().isoformat(), bitfile=str(bitfile),
                fpga_version='0xA7', feature_version=hex(pf.FEATURE_VERSION),
                configuration=fra._configuration(ol), rf_settings=pf.rf_settings(ol),
                setup=setup_info, amplitude_codes=amplitude_codes,
                phase_amplitude_rad=phase_amplitude_rad,
                minimum_measure_s=minimum_measure_s, minimum_settle_s=minimum_settle_s,
                allow_clipping=allow_clipping, repeats=8, max_repeats=24, target_snr=30,
                frequencies_hz=frequencies_hz.tolist())
csv_path.with_suffix('.json').write_text(json.dumps(metadata, indent=2) + '\n')
try:
    rows = pf.sweep(ol, frequencies_hz, mode=measurement_mode,
                    amplitude_codes=amplitude_codes,
                    phase_amplitude_rad=phase_amplitude_rad,
                    repeats=8, max_repeats=24, target_snr=30,
                    minimum_measure_s=minimum_measure_s,
                    minimum_settle_s=minimum_settle_s,
                    allow_clipping=allow_clipping, csv_path=csv_path)
finally:
    pf.stop(ol)  # Removes only the test sine; preserves lock settings/RF carrier.
print('Saved:', csv_path.resolve())
print('Blocks:', csv_path.with_name(csv_path.stem + '_blocks.csv').resolve())

## Interpretation and plots

Phasors use a convention in which a propagation delay gives negative phase.

* **RF:** `H = Phi_rx / Phi_tx` in rad/rad. Fit `arg(H) = phi0 − 2π f tau` over a band with nearly linear phase. This includes converter and receive mixer/FIR/CORDIC response, but excludes the phase-to-error scaling, tracking/IIR and analog feedback controller. The four-clock source generator delay is removed by measuring the applied phase at the RF DAC word.
* **Closed:** `H = Phi / D` in rad per injected DAC code. `U` is total DAC command and `D` is the injected sine. With negative feedback, `S = U/D` and the full return ratio is `L = (D−U)/U`. Thus `L` includes the frequency discriminator and all feedback paths. The absolute Φ/D response and the RF response have different units; do not divide them as though they were identical measurements. `Phi/U` is the plant response up to the received-phase probe, and excludes the digital phase-to-error calculation.
* A small `U` at high loop gain makes L poorly determined even when Φ/D is clean; use `loop_reliable` separately. Phase margin must be read from **L at |L|=1**, not from Φ/D or a closed-loop bump. Inspect every unity-gain crossing and the high-frequency trend before concluding stability.
The signed digital discriminator response itself is `E/Phi = (U−D)/Phi`, in DAC codes/rad. With IIR bypass and low modulation frequency, its magnitude should be about 65.2 codes/rad for /160, multiplied by the selected tracking gain. Its phase includes the processing between the CORDIC and the error-output injection point.

Closed raw-phase mode uses the nominal commanded D. Integer sine quantization causes a small systematic amplitude error (about 0.2% for 32 codes with uniformly sampled generator phases, larger at very small amplitudes). When L is much smaller than unity, subtraction D−U magnifies that error; the legacy simultaneous E/U measurement avoids relying on nominal D. Repeat at another amplitude and use the legacy analyzer for a precise high-frequency tail. Repeat scatter does not include this systematic uncertainty.


In [ ]:
data = pd.read_csv(csv_path)
f = data.frequency_hz.to_numpy()
h = data.response_re.to_numpy() + 1j*data.response_im.to_numpy()
reliable = data.reliable.to_numpy(dtype=bool)
finite = np.isfinite(h.real) & np.isfinite(h.imag) & (np.abs(h) > 0)
phase_deg = np.full(len(h), np.nan)
phase_deg[finite] = np.degrees(np.unwrap(np.angle(h[finite])))
fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax[0].semilogx(f[finite], 20*np.log10(abs(h[finite])), '.-')
ax[1].semilogx(f, phase_deg, '.-')
bad = finite & ~reliable
ax[0].scatter(f[bad], 20*np.log10(abs(h[bad])), marker='x', color='r', label='Unreliable')
ax[0].set_ylabel('20 log10 |H| (dB re ' + data.response_units.iloc[0] + ')')
ax[1].set_ylabel('Phase (deg)'); ax[1].set_xlabel('Modulation frequency (Hz)')
for a in ax: a.grid(True, which='both', alpha=.3)
ax[0].set_title(f'{measurement_mode}: received-phase response')
if np.any(bad): ax[0].legend()
fig.tight_layout(); fig.savefig(csv_path.with_suffix('.png'), dpi=160)
plt.show()

if measurement_mode == 'rf':
    fit_band = reliable & finite & (f >= 1e4) & (f <= 1e6)
    if fit_band.sum() >= 3:
        # Inverse phase-uncertainty weights: sigma_phi approximately SEM/|H|.
        weights = np.abs(h[fit_band])/np.maximum(data.complex_sem.to_numpy()[fit_band], 1e-15)
        slope, intercept = np.polyfit(f[fit_band], np.radians(phase_deg[fit_band]), 1, w=weights)
        delay_s = -slope/(2*np.pi)
        residual_deg = phase_deg[fit_band] - np.degrees(slope*f[fit_band] + intercept)
        print(f'RF phase-response delay, 10 kHz–1 MHz: {delay_s*1e9:.2f} ns')
        print(f'Phase-fit residual RMS: {np.std(residual_deg):.3f} deg')
        metadata['delay_fit'] = dict(minimum_hz=1e4, maximum_hz=1e6, delay_s=delay_s,
                                    intercept_rad=intercept, residual_rms_deg=float(np.std(residual_deg)))
        csv_path.with_suffix('.json').write_text(json.dumps(metadata, indent=2) + '\n')
else:
    loop = data.loop_re.to_numpy() + 1j*data.loop_im.to_numpy()
    sensitivity = data.sensitivity_re.to_numpy() + 1j*data.sensitivity_im.to_numpy()
    good = data.loop_reliable.to_numpy(dtype=bool) & np.isfinite(loop.real) & (abs(loop)>0)
    fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    ax[0].semilogx(f[good], 20*np.log10(abs(loop[good])), '.-', label='Return ratio L')
    ax[0].semilogx(f, 20*np.log10(np.maximum(abs(sensitivity), 1e-15)), '--', label='Sensitivity S')
    ax[0].axhline(0, color='k', lw=.7); ax[0].legend()
    ax[1].semilogx(f[good], np.degrees(np.unwrap(np.angle(loop[good]))), '.-')
    ax[1].axhline(-180, color='k', lw=.7)
    ax[0].set_ylabel('Magnitude (dB)'); ax[1].set_ylabel('Phase of L (deg)')
    ax[1].set_xlabel('Modulation frequency (Hz)')
    for a in ax: a.grid(True, which='both', alpha=.3)
    fig.tight_layout(); fig.savefig(csv_path.with_name(csv_path.stem + '_loop.png'), dpi=160)
    plt.show()

## Stop or finish

Stopping a measurement removes the probe using its normal ramp and leaves the operating point intact. The sweep already does this on success, exception or interrupt. The RF carrier is a separate setting: optionally turn it off after an RF test. Do not run RF setup on an optical lock you intend to retain.

In [ ]:
pf.stop(ol)
# Optional, only after the RF-chain test:
# if measurement_mode == 'rf':
#     ol.pll_rf_source.write(0, 0)   # RF carrier off; primary error remains DC zero.